# QC visualisations — box plot + missing values (MCP tool calls)

Reproduces the two calls used in the chat session of 2026-05-19 to fetch
rendered visualisation JSON from the MCP for a workspace seeded against
the **CKD Demo 2025** upload.

**Calls demonstrated**

1. `render_module_visualisation` for a placed `box_plot` module — **works**, returns full Plotly JSON.
2. `render_module_visualisation` for a placed `missing_values_by_sample_plot` module — **fails** with HTTP 400 (`Visualisation not supported for module type 'missing_values_by_sample_plot'`). This is an upstream visualisations-service coverage gap, not a md-python bug; see `memory/project_mcp_render_gap_missing_values_by_sample.md`.
3. **Client-side fallback** for (2): download `protein_intensity` from the raw INTENSITY dataset via the md_python client and compute the per-sample missing-value summary directly.

Each MCP tool function is imported and called directly. The `@mcp.tool()` decorator in FastMCP registers the callable with the server while leaving the function itself callable as a normal Python function — so what runs in this notebook is exactly the code the MCP server runs in response to a tool-use request.

## Fixed identifiers from the session

| Thing | UUID |
|---|---|
| Workspace `test visualization` | `bb544a53-de85-4a7c-b782-a441357b4573` |
| Tab `QC` | `26e89fec-a49f-4524-834c-d76f7f90ada7` |
| Module: `box_plot` on NI dataset | `83fbdefa-717e-4aad-844c-37e9a05ef501` |
| Module: `missing_values_by_sample_plot` on raw | `eddc9007-8ed0-40b7-889a-853cd2837364` |
| NI dataset (`CKD demo imputation`) | `fb3e4141-926e-4b97-bf20-e29d6d818570` |
| Raw INTENSITY dataset (`CKD Demo 2025`) | `1e97e2ae-1ad6-4d3b-a772-f2d496d152fa` |
| Upload (`CKD Demo 2025`) | `6db6e841-a1ef-4d74-b920-c6be7b581fb1` |

## Setup

Loads `.env` (same way `mcp_server.py` does — resolves relative to the repo root, not the notebook's cwd) and pulls in the MCP tool functions.

In [ ]:
import json
from pathlib import Path

from mcp_tools._env import load_env_from

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "mcp_server.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "mcp_server.py").exists(), "Could not find repo root (looked for mcp_server.py)"
load_env_from(REPO_ROOT)

from mcp_tools.workspaces.modules import render_module_visualisation
from mcp_tools.datasets.download import download_dataset_table

WORKSPACE_ID = "bb544a53-de85-4a7c-b782-a441357b4573"
TAB_ID = "26e89fec-a49f-4524-834c-d76f7f90ada7"
BOX_PLOT_MODULE_ID = "83fbdefa-717e-4aad-844c-37e9a05ef501"
MISSING_VALUES_MODULE_ID = "eddc9007-8ed0-40b7-889a-853cd2837364"
RAW_INTENSITY_DATASET_ID = "1e97e2ae-1ad6-4d3b-a772-f2d496d152fa"

print("Repo root:", REPO_ROOT)

## 1. Box plot — works

Mirrors the chat-session MCP call:

```
render_module_visualisation(
    workspace_id=WORKSPACE_ID,
    tab_id=TAB_ID,
    module_id=BOX_PLOT_MODULE_ID,
)
```

Returns a JSON string (the Plotly figure). Parse with `json.loads`.

In [ ]:
raw = render_module_visualisation(
    workspace_id=WORKSPACE_ID,
    tab_id=TAB_ID,
    module_id=BOX_PLOT_MODULE_ID,
)

if raw.startswith("Error:"):
    print(raw)
    box_fig = None
else:
    box_fig = json.loads(raw)
    print("Title:", box_fig["layout"]["title"]["text"])
    box_traces = [t for t in box_fig["data"] if t["type"] == "box"]
    print("Box traces (one per sample):", len(box_traces))
    print("First trace name:", box_traces[0]["name"])

In [ ]:
# Tabulate per-sample medians / means / IQR — same numbers the rendered
# plot shows, just in a dataframe so you can sort and eyeball outliers.
import pandas as pd

rows = []
for t in box_traces:
    rows.append(
        {
            "sample": t["name"],
            "median": t["median"][0],
            "mean": t["mean"][0],
            "q1": t["q1"][0],
            "q3": t["q3"][0],
            "lower_fence": t["lowerfence"][0],
            "upper_fence": t["upperfence"][0],
        }
    )

df_box = pd.DataFrame(rows)
df_box["sample"] = df_box["sample"].str.replace("LFQ intensity ", "", regex=False)
df_box.sort_values("median", ascending=False).head(10)

## 2. Missing values by sample — MCP render fails

Same call shape as above, different module id. Expect a `400`:

```
Error: Failed to render visualisation: 400 - {"error": "Visualisation not supported for module type 'missing_values_by_sample_plot'"}
```

The module itself is correctly placed and renders fine in the workspace UI — the gap is specifically in the out-of-band render route used by `render_module_visualisation`.

In [ ]:
mv_raw = render_module_visualisation(
    workspace_id=WORKSPACE_ID,
    tab_id=TAB_ID,
    module_id=MISSING_VALUES_MODULE_ID,
)
print(mv_raw)

## 3. Client-side fallback for missing values

Downloads `protein_intensity` from the raw INTENSITY dataset via the
`download_dataset_table` MCP tool (URL mode), reads it with pandas, and
computes the per-sample completeness — the same number the bar chart
would have shown.

This is the recommended workaround until the visualisations-service registers an out-of-band render route for `missing_values_by_sample_plot`.

In [ ]:
url_envelope = json.loads(
    download_dataset_table(
        dataset_id=RAW_INTENSITY_DATASET_ID,
        table_name="protein_intensity",
        format="csv",
    )
)
url_envelope

In [ ]:
intensity_df = pd.read_csv(url_envelope["download_url"])
print("Shape:", intensity_df.shape)
print("First 3 columns:", intensity_df.columns[:3].tolist())
print("Last 5 columns:", intensity_df.columns[-5:].tolist())
intensity_df.head(3)

In [ ]:
# Sample intensity columns start with "LFQ intensity " in this dataset.
# Adjust the prefix if the dataset uses a different convention.
sample_cols = [c for c in intensity_df.columns if c.startswith("LFQ intensity ")]
print(f"Detected {len(sample_cols)} sample columns")

n_features = len(intensity_df)
missing = intensity_df[sample_cols].isna().sum()
completeness = pd.DataFrame(
    {
        "sample": [c.replace("LFQ intensity ", "") for c in sample_cols],
        "n_present": (n_features - missing).values,
        "n_missing": missing.values,
        "pct_present": (100 * (n_features - missing) / n_features).round(2).values,
    }
)
completeness.sort_values("pct_present").head(10)

In [ ]:
# Top of the distribution — best-detected samples.
completeness.sort_values("pct_present", ascending=False).head(10)

In [ ]:
# Quick visual summary equivalent to what the bar chart would render.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
plot_df = completeness.sort_values("sample")
ax.bar(plot_df["sample"], plot_df["pct_present"])
ax.set_ylabel("% features detected")
ax.set_xlabel("sample")
ax.set_title("Per-sample protein detection — CKD Demo 2025 (raw)")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()